In [0]:
#Step: Extracting the data from the API

import requests
import json

#calling the API
api_key = dbutils.secrets.get(scope="nasa-scope", key="api-nasa-key")
url = f"https://api.nasa.gov/neo/rest/v1/neo/browse?api_key={api_key}"

# requesting the data
response = requests.get(url)

# verification of the request
if response.status_code == 200:
    data = response.json()
    print("Request successful!")
    
    from pyspark.sql.functions import explode, col
# Normalize the JSON data to flatten nested structures
    if 'near_earth_objects' in data:
        neo_data = data['near_earth_objects']
        df_bronze = spark.read.json(sc.parallelize([json.dumps(neo_data)]))
    else:
        df_bronze = spark.createDataFrame([data])

        display(df_bronze)


In [0]:

account= "nasapip"
secret = "kvsecret"

storage_key = dbutils.secrets.get(scope="nasa-scope", key="kvsecret")

# 3. Spark Authentication
spark.conf.set(
    f"fs.azure.account.key.{account}.dfs.core.windows.net",
    storage_key
)

In [0]:
#Storing the data as a bronze file

# container path
bronze_path= f"abfss://bronze@{account}.dfs.core.windows.net/asteroides_neo_raw/"

df_bronze.write.mode("append").format("parquet").save(bronze_path)

print(f"data downloaded successfully in: {bronze_path}")

In [0]:
#Step: Transforming the data from bronze to silver

from pyspark.sql.functions import current_timestamp

silver_path = f"abfss://silver@{account}.dfs.core.windows.net/asteroides_neo_cleaned/"

print("Reading the data")
df_silver_raw = spark.read.format("parquet").load(bronze_path)

print("cleaning and adding metadata")
#Audit Column
df_clean_silver= df_silver_raw.withColumn("silver_processament_data", current_timestamp())

display(df_clean_silver)

print("Download on the silver layer successfully.")

df_clean_silver.write.mode("overwrite").format("parquet").save(silver_path)

print(f"Saved on the silver layer successfull in: {silver_path}")

In [0]:
#Step: Transforming the data from silver to gold

from pyspark.sql.functions import col

gold_path = f"abfss://gold@{account}.dfs.core.windows.net/hazardous_asteroid/"

print("Reading cleaned data...")
df_silver = spark.read.format("parquet").load(silver_path)

print("Applying business rules...")

df_gold = df_silver.filter(col("is_potentially_hazardous_asteroid") == True)

df_gold_final = df_gold.select(
    "id",
    "name",
    "is_potentially_hazardous_asteroid",
    "silver_processament_data"
)

display(df_gold_final)

print("Saving data to gold layer...")
df_gold_final.write.mode("overwrite").format("parquet").save(gold_path)

print(f"Data saved successfully in: {gold_path}")